# NB1A - Pokemon Data Collection
This notebook contains the code used to collect Pokémon data from the PokeAPI. 

The official PokeAPI documentation can be found here: [PokeAPI Documentation](https://pokeapi.co/docs/v2).

**Note:** Some parts of the documentation are outdated or inaccurate. These issues will be highlighted when encountered.

## Importing Required Libraries

We will import the necessary libraries to handle API requests, JSON processing, and data manipulation.


In [8]:
import json
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

## Function to Retrieve Pokémon Data by Generation

The function below retrieves all Pokémon species for a given generation using the PokeAPI. 

- It makes a request to the API using the **generation number**.
- It extracts the Pokémon species data and normalizes it into a Pandas DataFrame.
- A new column is added to store the **generation number**.


In [9]:
def get_generations_pokemon(generation: int):
    request_url = f'https://pokeapi.co/api/v2/generation/{generation}'
    response = requests.get(request_url)
    data = response.json()
    pokemon_df = pd.json_normalize(data, record_path = 'pokemon_species')
    pokemon_df['generation'] = generation
    return pokemon_df

## Collecting Pokémon Data for All Generations

We iterate through all Pokémon generations (1 to 9) and retrieve their species data.

- The data is stored in a list of DataFrames.
- These DataFrames are concatenated into a single DataFrame (`poke_df`).
- The resulting table is displayed.


In [10]:
list_of_generations = [1, 2, 3, 4, 5, 6, 7, 8, 9]
list_of_df = [get_generations_pokemon(generation) for generation in list_of_generations]
poke_df = pd.concat(list_of_df)
display(poke_df)


,name,url,generation
0,bulbasaur,https://pokeapi.co/api/v2/pokemon-species/1/,1
1,charmander,https://pokeapi.co/api/v2/pokemon-species/4/,1
2,squirtle,https://pokeapi.co/api/v2/pokemon-species/7/,1
3,caterpie,https://pokeapi.co/api/v2/pokemon-species/10/,1
4,weedle,https://pokeapi.co/api/v2/pokemon-species/13/,1
...,...,...,...
115,gholdengo,https://pokeapi.co/api/v2/pokemon-species/1000/,9
116,dipplin,https://pokeapi.co/api/v2/pokemon-species/1011/,9
117,sinistcha,https://pokeapi.co/api/v2/pokemon-species/1013/,9
118,archaludon,https://pokeapi.co/api/v2/pokemon-species/1018/,9


## Extracting Pokémon ID from API URLs

The Pokémon species data includes a URL containing the Pokémon ID. 

The function below extracts the **Pokédex ID** from the URL and adds it as a new column in our DataFrame.


In [11]:
def extract_pokemon_id(url): 
    id = url.rstrip('/').split('/')[-1]
    return int(id)

poke_df['pokemon_id'] = poke_df['url'].apply(extract_pokemon_id)
poke_df = poke_df.sort_values('pokemon_id').reset_index(drop = True)
display(poke_df)


,name,url,generation,pokemon_id
0,bulbasaur,https://pokeapi.co/api/v2/pokemon-species/1/,1,1
1,ivysaur,https://pokeapi.co/api/v2/pokemon-species/2/,1,2
2,venusaur,https://pokeapi.co/api/v2/pokemon-species/3/,1,3
3,charmander,https://pokeapi.co/api/v2/pokemon-species/4/,1,4
4,charmeleon,https://pokeapi.co/api/v2/pokemon-species/5/,1,5
...,...,...,...,...
1020,raging-bolt,https://pokeapi.co/api/v2/pokemon-species/1021/,9,1021
1021,iron-boulder,https://pokeapi.co/api/v2/pokemon-species/1022/,9,1022
1022,iron-crown,https://pokeapi.co/api/v2/pokemon-species/1023/,9,1023
1023,terapagos,https://pokeapi.co/api/v2/pokemon-species/1024/,9,1024


## Saving Pokémon Data to JSON

The collected Pokémon species data is saved as a JSON file in the `generation_pokemon` directory.

In [12]:
poke_df.to_json('../../data/pokemon_data/generation_pokemon/generation_pokemon.json')

## Function to Collect Detailed Pokémon Data

This function retrieves additional details for each Pokémon:

1. **Basic Details:**
   - Extracts **sprite images** (removes extra sprite data).
   - Saves the data as JSON in the `pokemon_details` directory.

2. **Species Details:**
   - Retrieves species information such as the **flavor text** (Pokédex description).
   - Stores only the English description.
   - Saves the data as JSON in the `species_details` directory.

Any errors encountered during the request process are handled and logged.


In [13]:
def collect_pokemon_data(pokemon_id):
    try: 
        details_url = f'https://pokeapi.co/api/v2/pokemon/{pokemon_id}'
        details_response = requests.get(details_url)
        if details_response.status_code == 200:
            details_data = details_response.json()
            details_data['pokemon_portrait'] = details_data['sprites']['front_default']
            details_data.pop('sprites')
            with open(f'../../data/pokemon_data/pokemon_details/pokemon_{pokemon_id}_details.json', 'w') as file:
                json.dump(details_data, file)

        species_url = f'https://pokeapi.co/api/v2/pokemon-species/{pokemon_id}'
        species_response = requests.get(species_url)
        if species_response.status_code == 200:
            species_data = species_response.json()
            english_entry = next(entry for entry in species_data['flavor_text_entries'] if entry['language']['name'] == 'en')
            species_data['flavor_text_entries'] = english_entry
            with open(f'../../data/pokemon_data/species_details/pokemon_{pokemon_id}_species_details.json', 'w') as file:
                json.dump(species_data, file)       

        return 
    
    except Exception as e: 
        return f'Error for ID {pokemon_id}: {e}'

## Running Pokémon Data Collection in Parallel

To speed up the collection process, we use **ThreadPoolExecutor**, allowing multiple API requests to run in parallel.

In [14]:
def main(pokemon_ids): 
    with ThreadPoolExecutor() as executor: 
        results = list(executor.map(collect_pokemon_data, pokemon_ids))
    return results

## Running Data Collection for All Pokémon

This script:
- Defines a list of Pokémon IDs (from **1 to 1025**).
- Calls the `main()` function to retrieve data in parallel.
- Displays the number of Pokémon for which data was successfully obtained.

In [ ]:
pokemon_ids = list(range(1,1026))
results = main(pokemon_ids)
print(f'Data successfully obtained for {results.count(None)} Pokemon.')